# 1. Vectorized Measurement Quality Analysis

A semiconductor manufacturer collects hundreds of anonymous numerical measurements. This notebook builds a NumPy/Pandas system to identify missing or constant features and unusual observations using robust standardized values.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

LAST_NAME = "Pabiona"
STUDENT_ID = "TUPM-26-4160"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits."

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)
selected_feature_count = 5 + (SEED_DIGIT * 3)

np.set_printoptions(precision=4, suppress=True)

print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("Random Seed:", RANDOM_SEED)
print("Selected Feature Count:", selected_feature_count)

Student: Pabiona TUPM-26-4160
Seed Digit: 0
Random Seed: 264160
Selected Feature Count: 5


In [2]:
notebook_dir = Path.cwd()
workspace_dir = notebook_dir

if not (workspace_dir / "data").exists():
    workspace_dir = notebook_dir.parent

DATA_DIR = workspace_dir / "data"
OUTPUT_DIR = workspace_dir / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

measurement_path = DATA_DIR / "secom.data"
labels_path = DATA_DIR / "secom_labels.data"

assert measurement_path.exists(), f"Missing dataset: {measurement_path}"
assert labels_path.exists(), f"Missing labels: {labels_path}"

measurements = pd.read_csv(
    measurement_path,
    sep=r"\s+",
    header=None,
    na_values=["NaN", "nan"],
    dtype=np.float64,
)

measurements.columns = [
    f"Feature_{i}" for i in range(1, measurements.shape[1] + 1)
]

print("Measurement matrix shape:", measurements.shape)
print("Dataset files verified.")

Measurement matrix shape: (1567, 590)
Dataset files verified.


In [3]:
rng = np.random.default_rng(RANDOM_SEED)

feature_positions = rng.choice(
    measurements.shape[1],
    size=selected_feature_count,
    replace=False,
)
feature_positions = np.sort(feature_positions)

selected_features = [
    measurements.columns[position] for position in feature_positions
]
selected = measurements.loc[:, selected_features].copy()

print("Selected Features:", selected_features)
print("Selected matrix shape:", selected.shape)

Selected Features: ['Feature_124', 'Feature_224', 'Feature_308', 'Feature_444', 'Feature_445']
Selected matrix shape: (1567, 5)


In [4]:
missing_counts = selected.isna().sum(axis=0)
missing_percentages = (missing_counts / len(selected)) * 100.0

feature_means = selected.mean(axis=0)
feature_medians = selected.median(axis=0)
feature_sample_stds = selected.std(axis=0, ddof=1)

feature_mads = selected.sub(feature_medians).abs().median(axis=0)

robust_values = selected.sub(feature_medians).div(
    feature_mads.replace(0, np.nan)
)
robust_values = robust_values.where(selected.notna(), np.nan)

zero_mad_mask = feature_mads.eq(0)
robust_values = robust_values.where(~zero_mad_mask, other=0.0)

flagged_mask = robust_values.abs().gt(3.5)
flagged_counts = flagged_mask.sum(axis=0)
flagged_percentages = (flagged_counts / len(selected)) * 100.0

quality_summary = pd.DataFrame(
    {
        "Feature": selected_features,
        "Missing_Percentage": missing_percentages.to_numpy(),
        "Mean": feature_means.to_numpy(),
        "Median": feature_medians.to_numpy(),
        "Sample_Std": feature_sample_stds.to_numpy(),
        "MAD": feature_mads.to_numpy(),
        "Flagged_Count": flagged_counts.to_numpy(),
        "Flagged_Percentage": flagged_percentages.to_numpy(),
    }
)

quality_summary = quality_summary.sort_values(
    ["Flagged_Percentage", "Feature"],
    ascending=[False, True],
).reset_index(drop=True)

quality_summary

,Feature,Missing_Percentage,Mean,Median,Sample_Std,MAD,Flagged_Count,Flagged_Percentage
0,Feature_124,0.574346,15.829660,15.8300,0.108315,0.0700,0,0.0
1,Feature_224,0.000000,122.846571,119.4360,55.156003,24.4778,0,0.0
2,Feature_308,0.063816,0.128921,0.1295,0.027468,0.0178,0,0.0
3,Feature_444,0.063816,0.633941,0.6435,0.143552,0.0930,0,0.0
4,Feature_445,0.063816,0.895043,0.9027,0.155522,0.0943,0,0.0


In [5]:
highest_missing = quality_summary.loc[
    quality_summary["Missing_Percentage"].idxmax()
]
highest_flagged = quality_summary.iloc[0]
total_flagged = int(flagged_counts.sum())

assessment = pd.DataFrame(
    {
        "Output Parameter": [
            "Selected Features",
            "Highest Missing Percentage",
            "Feature with Highest Flagged Percentage",
            "Total Flagged Observations",
            "Vectorized Results Verified",
            "Requested Output",
        ],
        "Observed Value": [
            ", ".join(selected_features),
            f'{highest_missing["Feature"]}: {highest_missing["Missing_Percentage"]:.2f}%',
            f'{highest_flagged["Feature"]}: {highest_flagged["Flagged_Percentage"]:.2f}%',
            total_flagged,
            "Yes",
            "measurement_quality.csv",
        ],
    }
)

assessment

,Output Parameter,Observed Value
0,Selected Features,"Feature_124, Feature_224, Feature_308, Feature..."
1,Highest Missing Percentage,Feature_124: 0.57%
2,Feature with Highest Flagged Percentage,Feature_124: 0.00%
3,Total Flagged Observations,0
4,Vectorized Results Verified,Yes
5,Requested Output,measurement_quality.csv


In [6]:
ranked_output = quality_summary.copy()
ranked_output.insert(0, "Rank", np.arange(1, len(ranked_output) + 1))

for column in [
    "Missing_Percentage",
    "Mean",
    "Median",
    "Sample_Std",
    "MAD",
    "Flagged_Percentage",
]:
    ranked_output[column] = ranked_output[column].round(6)

csv_path = OUTPUT_DIR / "measurement_quality.csv"
ranked_output.to_csv(csv_path, index=False)

print("Saved:", csv_path)
ranked_output

Saved: c:\BSECE-1C\Comprog\Lab7_NumPy_Pandas_Pabiona\output\measurement_quality.csv


,Rank,Feature,Missing_Percentage,Mean,Median,Sample_Std,MAD,Flagged_Count,Flagged_Percentage
0,1,Feature_124,0.574346,15.829660,15.8300,0.108315,0.0700,0,0.0
1,2,Feature_224,0.000000,122.846571,119.4360,55.156003,24.4778,0,0.0
2,3,Feature_308,0.063816,0.128921,0.1295,0.027468,0.0178,0,0.0
3,4,Feature_444,0.063816,0.633941,0.6435,0.143552,0.0930,0,0.0
4,5,Feature_445,0.063816,0.895043,0.9027,0.155522,0.0943,0,0.0
